# 05. Evaluacion de retrieval

Este notebook compara la busqueda `keyword` y la busqueda `semantic` sobre un conjunto pequeno de consultas curadas para el proyecto.

## Objetivos

- cargar los splits `dev`, `val` y `test`
- ejecutar evaluacion reutilizando `src.evaluation`
- comparar `precision@k`, `recall@k`, `MRR` y `hit@k`
- exportar resultados detallados y resumen agregado a `outputs/`


In [6]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.data_loader import load_evaluation_queries
from src.evaluation import (
    evaluate_keyword_search,
    evaluate_semantic_search,
    results_to_frame,
    summarize_results,
)

OUTPUTS_DIR = ROOT / "outputs"
RESULTS_PATH = OUTPUTS_DIR / "evaluation_results.csv"
SUMMARY_PATH = OUTPUTS_DIR / "evaluation_summary.csv"
SPLITS = ("dev", "val", "test")
TOP_K = 5

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("RESULTS_PATH:", RESULTS_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)
print("TOP_K:", TOP_K)


ROOT: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03
RESULTS_PATH: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_results.csv
SUMMARY_PATH: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_summary.csv
TOP_K: 5


## 1. Inspeccionar queries de evaluacion

Cada split contiene consultas curadas con uno o mas `CUCE` relevantes.


In [7]:
split_frames = []
for split in SPLITS:
    frame = load_evaluation_queries(split)
    frame = frame.copy()
    frame["split"] = split
    split_frames.append(frame)
    print(f"{split}: {len(frame)} queries")

queries_frame = pd.concat(split_frames, ignore_index=True)
display(queries_frame)


dev: 6 queries
val: 3 queries
test: 3 queries


,query_id,query_text,relevant_cuce,metadata_filters,notes,categoria,split
0,dev_001,medicamentos para hospital en santa cruz,26-0417-03-1669697-1-1,NaN,Consulta farmacologica hospitalaria,medicamentos,dev
1,dev_002,reactivos de laboratorio clinico para hospital,"26-0902-21-1669603-1-1,26-1705-00-1669336-1-1",NaN,Reactivos de laboratorio hospitalario,reactivos,dev
2,dev_003,mantenimiento de vias urbanas con cemento en t...,26-1519-00-1669672-1-1,NaN,Caso de mantenimiento urbano,infraestructura,dev
3,dev_004,software libre para gestion clinica en salud,26-0046-38-1660991-1-1,NaN,Caso unico de software en salud,software,dev
4,dev_005,alcantarillado pluvial en la paz,26-1201-00-1668159-1-1,NaN,Consulta de obra publica urbana,obras,dev
5,dev_006,equipamiento sub alcaldia tupiza papel,26-1519-00-1669618-1-1,NaN,Compra administrativa de equipamiento,bienes,dev
6,val_001,amoxicilina de uso hospitalario para farmacia,26-1101-04-1669637-1-1,NaN,Medicamento hospitalario especifico,medicamentos,val
7,val_002,rescate en aeronaves e incendios estructurales...,26-0389-00-1669576-1-1,NaN,Equipos especializados de rescate,bienes,val
8,val_003,alcantarillado sanitario y pluvial con cemento...,26-1206-00-1669479-1-1,NaN,Mantenimiento sanitario con insumo principal,infraestructura,val
9,test_001,medicamentos e insumos para centro de salud de...,26-1219-00-1669640-1-1,NaN,Consulta municipal de abastecimiento,medicamentos,test


## 2. Ejecutar evaluacion comparativa

Se calcula evaluacion separada por split y por metodo usando la capa productiva de retrieval.


In [8]:
detail_frames = []
summary_rows = []

for split in SPLITS:
    keyword_results = evaluate_keyword_search(split=split, k=TOP_K)
    semantic_results = evaluate_semantic_search(split=split, k=TOP_K)

    keyword_frame = results_to_frame(keyword_results)
    keyword_frame["split"] = split
    semantic_frame = results_to_frame(semantic_results)
    semantic_frame["split"] = split

    detail_frames.extend([keyword_frame, semantic_frame])

    keyword_summary = summarize_results(keyword_results)
    semantic_summary = summarize_results(semantic_results)

    summary_rows.extend(
        [
            {
                "split": split,
                "method": keyword_summary.method,
                "query_count": keyword_summary.query_count,
                "k": keyword_summary.k,
                "mean_precision_at_k": keyword_summary.mean_precision_at_k,
                "mean_recall_at_k": keyword_summary.mean_recall_at_k,
                "mean_reciprocal_rank": keyword_summary.mean_reciprocal_rank,
                "hit_rate_at_k": keyword_summary.hit_rate_at_k,
            },
            {
                "split": split,
                "method": semantic_summary.method,
                "query_count": semantic_summary.query_count,
                "k": semantic_summary.k,
                "mean_precision_at_k": semantic_summary.mean_precision_at_k,
                "mean_recall_at_k": semantic_summary.mean_recall_at_k,
                "mean_reciprocal_rank": semantic_summary.mean_reciprocal_rank,
                "hit_rate_at_k": semantic_summary.hit_rate_at_k,
            },
        ]
    )

details_frame = pd.concat(detail_frames, ignore_index=True)
summary_frame = pd.DataFrame(summary_rows)

details_frame.to_csv(RESULTS_PATH, index=False)
summary_frame.to_csv(SUMMARY_PATH, index=False)

print("Resultados detallados guardados en:", RESULTS_PATH)
print("Resumen agregado guardado en:", SUMMARY_PATH)


Resultados detallados guardados en: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_results.csv
Resumen agregado guardado en: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_summary.csv


## 3. Resumen agregado


In [9]:
display(summary_frame.sort_values(["split", "method"]).reset_index(drop=True))

summary_pivot = summary_frame.pivot(index="split", columns="method", values=[
    "mean_precision_at_k",
    "mean_recall_at_k",
    "mean_reciprocal_rank",
    "hit_rate_at_k",
])
display(summary_pivot)


,split,method,query_count,k,mean_precision_at_k,mean_recall_at_k,mean_reciprocal_rank,hit_rate_at_k
0,dev,keyword,6,5,0.000000,0.000000,0.000000,0.000000
1,dev,semantic,6,5,0.033333,0.166667,0.166667,0.166667
2,test,keyword,3,5,0.000000,0.000000,0.000000,0.000000
3,test,semantic,3,5,0.066667,0.333333,0.333333,0.333333
4,val,keyword,3,5,0.000000,0.000000,0.000000,0.000000
5,val,semantic,3,5,0.066667,0.333333,0.333333,0.333333


mean_precision_at_k           mean_recall_at_k            \
method             keyword  semantic          keyword  semantic   
split                                                             
dev                    0.0  0.033333              0.0  0.166667   
test                   0.0  0.066667              0.0  0.333333   
val                    0.0  0.066667              0.0  0.333333   

       mean_reciprocal_rank           hit_rate_at_k            
method              keyword  semantic       keyword  semantic  
split                                                          
dev                     0.0  0.166667           0.0  0.166667  
test                    0.0  0.333333           0.0  0.333333  
val                     0.0  0.333333           0.0  0.333333

## 4. Diagnostico por consulta

Esta tabla permite revisar en que consultas falla cada metodo y comparar los `CUCE` recuperados.


In [10]:
diagnostic_columns = [
    "split",
    "query_id",
    "query_text",
    "method",
    "relevant_cuces",
    "retrieved_cuces",
    "precision_at_k",
    "recall_at_k",
    "reciprocal_rank",
    "hit_at_k",
]

display(
    details_frame[diagnostic_columns]
    .sort_values(["split", "query_id", "method"])
    .reset_index(drop=True)
)


,split,query_id,query_text,method,relevant_cuces,retrieved_cuces,precision_at_k,recall_at_k,reciprocal_rank,hit_at_k
0,dev,dev_001,medicamentos para hospital en santa cruz,keyword,26-0417-03-1669697-1-1,,0.0,0.0,0.0,0.0
1,dev,dev_001,medicamentos para hospital en santa cruz,semantic,26-0417-03-1669697-1-1,"26-0902-21-1668824-1-1,26-1101-04-1669637-1-1,...",0.0,0.0,0.0,0.0
2,dev,dev_002,reactivos de laboratorio clinico para hospital,keyword,"26-0902-21-1669603-1-1,26-1705-00-1669336-1-1",,0.0,0.0,0.0,0.0
3,dev,dev_002,reactivos de laboratorio clinico para hospital,semantic,"26-0902-21-1669603-1-1,26-1705-00-1669336-1-1","26-0901-02-1669175-1-1,26-0902-43-1667329-1-1,...",0.0,0.0,0.0,0.0
4,dev,dev_003,mantenimiento de vias urbanas con cemento en t...,keyword,26-1519-00-1669672-1-1,,0.0,0.0,0.0,0.0
5,dev,dev_003,mantenimiento de vias urbanas con cemento en t...,semantic,26-1519-00-1669672-1-1,,0.0,0.0,0.0,0.0
6,dev,dev_004,software libre para gestion clinica en salud,keyword,26-0046-38-1660991-1-1,,0.0,0.0,0.0,0.0
7,dev,dev_004,software libre para gestion clinica en salud,semantic,26-0046-38-1660991-1-1,"26-0046-38-1660991-1-1,26-1704-00-1668525-1-1,...",0.2,1.0,1.0,1.0
8,dev,dev_005,alcantarillado pluvial en la paz,keyword,26-1201-00-1668159-1-1,,0.0,0.0,0.0,0.0
9,dev,dev_005,alcantarillado pluvial en la paz,semantic,26-1201-00-1668159-1-1,26-0417-09-1669039-1-1,0.0,0.0,0.0,0.0


## Notas

- Este set de queries es pequeno y curado; sirve para validacion inicial y comparacion metodologica.
- Si se ajusta el embedding, el notebook `04` debe reejecutarse antes de volver a medir retrieval.
- El siguiente paso natural es ampliar el dataset de queries y evaluar una estrategia hibrida.
